# 🚗 DriverPulse — Namma Yatri Ward-Level EDA

**Objective**: Clean, parse, and explore ward-level ride data from Namma Yatri (Bengaluru).  
Engineer business-relevant features and export a cleaned dataset for modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded ✅')

## 1. Load Raw Data

In [ ]:
raw = pd.read_csv('data/namma_yatri_bengaluru_ward_wise_all_time_data.csv')
print(f'Shape: {raw.shape}')
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.tail()

## 2. Data Cleaning & Parsing

The dataset has:
- **Indian-format numbers** (`1,23,456`) — commas must be stripped before converting to int/float
- **Currency symbols** (`₹`) on earnings/fare columns
- **Percentage strings** (`93.5%`) on rate columns
- A potential trailing empty row

In [ ]:
df = raw.copy()

# Drop rows where Ward is NaN or empty
df = df.dropna(subset=['Ward'])
df = df[df['Ward'].str.strip() != '']
df = df.reset_index(drop=True)

print(f'Shape after dropping empty rows: {df.shape}')

In [ ]:
def parse_indian_number(val):
    """Parse Indian-formatted numbers: remove ₹, commas, then convert to float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    s = s.replace('₹', '').replace(',', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_percentage(val):
    """Parse percentage strings like '93.5%' → 0.935"""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace('%', '')
    try:
        return float(s) / 100.0
    except ValueError:
        return np.nan


# --- Define column groups ---
numeric_cols = [
    'Searches', 'Searches which got estimate', 'Searches for Quotes',
    'Searches which got Quotes', 'Bookings', 'Completed Trips',
    'Cancelled Bookings',
    "Driver's Earnings", 'Average Fare per Trip', 'Distance Travelled (km)'
]

rate_cols = [
    'Search-to-estimate Rate', 'Rider Fare Acceptance Rate',
    'Driver Quote Acceptance Rate', 'Quote-to-Booking Rate',
    'Booking Cancellation Rate', 'Driver Cancellation Rate',
    'User Cancellation Rate', 'Conversion Rate'
]

# Average Distance per Trip (km) is already numeric-ish but may have commas
distance_col = 'Average Distance per Trip (km)'

# Parse numeric columns
for col in numeric_cols:
    df[col] = df[col].apply(parse_indian_number)

# Parse distance column
df[distance_col] = df[distance_col].apply(parse_indian_number)

# Parse rate/percentage columns
for col in rate_cols:
    df[col] = df[col].apply(parse_percentage)

print('Parsing complete ✅')
df.dtypes

In [ ]:
# Quick sanity check — verify a known row
print('=== Sample: Koramangala ===')
display(df[df['Ward'] == 'Koramangala'])

In [ ]:
# Check for remaining nulls
null_summary = df.isnull().sum()
null_summary[null_summary > 0]

In [ ]:
df.describe().T

## 3. Feature Engineering

We create the following business-relevant derived features:

| Feature | Formula | Interpretation |
|---|---|---|
| `earnings_per_km` | earnings / distance | Revenue efficiency |
| `supply_gap` | searches − quotes given | Unmet demand |
| `revenue_leakage` | searches − (completed trips × avg fare) | Lost potential revenue |
| `reliability_score` | (1 − driver_cancel_rate) × driver_quote_acceptance_rate | Driver reliability |

In [ ]:
# Earnings per km
df['earnings_per_km'] = df["Driver's Earnings"] / df['Distance Travelled (km)']

# Supply gap: searches that never got quotes
df['supply_gap'] = df['Searches'] - df['Searches which got Quotes']

# Revenue leakage: searches minus revenue captured
df['revenue_leakage'] = df['Searches'] - (df['Completed Trips'] * df['Average Fare per Trip'])

# Reliability score
df['reliability_score'] = (1 - df['Driver Cancellation Rate']) * df['Driver Quote Acceptance Rate']

print('Engineered features added ✅')
df[['Ward', 'earnings_per_km', 'supply_gap', 'revenue_leakage', 'reliability_score']].head(10)

## 4. Rename Columns (snake_case for downstream ML)

Clean column names for easier use in modeling notebooks.

In [ ]:
rename_map = {
    'Ward': 'ward',
    'Searches': 'searches',
    'Searches which got estimate': 'searches_with_estimate',
    'Searches for Quotes': 'searches_for_quotes',
    'Searches which got Quotes': 'searches_with_quotes',
    'Bookings': 'bookings',
    'Completed Trips': 'completed_trips',
    'Search-to-estimate Rate': 'search_to_estimate_rate',
    'Rider Fare Acceptance Rate': 'rider_fare_acceptance_rate',
    'Driver Quote Acceptance Rate': 'driver_quote_acceptance_rate',
    'Quote-to-Booking Rate': 'quote_to_booking_rate',
    'Cancelled Bookings': 'cancelled_bookings',
    'Booking Cancellation Rate': 'booking_cancellation_rate',
    'Driver Cancellation Rate': 'driver_cancellation_rate',
    'User Cancellation Rate': 'user_cancellation_rate',
    'Conversion Rate': 'conversion_rate',
    "Driver's Earnings": 'driver_earnings',
    'Average Distance per Trip (km)': 'avg_distance',
    'Average Fare per Trip': 'avg_fare',
    'Distance Travelled (km)': 'distance_travelled_km',
}

df = df.rename(columns=rename_map)
print(f'Columns: {list(df.columns)}')

## 5. Exploratory Data Analysis

### 5.1 Distributions of Key Metrics

In [ ]:
dist_cols = [
    'completed_trips', 'driver_earnings', 'avg_fare', 'avg_distance',
    'conversion_rate', 'driver_cancellation_rate',
    'earnings_per_km', 'reliability_score'
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for i, col in enumerate(dist_cols):
    ax = axes[i]
    sns.histplot(df[col], kde=True, ax=ax, color=sns.color_palette('viridis', 8)[i], edgecolor='white')
    ax.set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.axvline(df[col].median(), color='red', linestyle='--', alpha=0.7, label=f'Median: {df[col].median():.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Distribution of Key Metrics Across Wards', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.2 Correlation Heatmap

In [ ]:
corr_cols = [
    'searches', 'bookings', 'completed_trips', 'conversion_rate',
    'driver_quote_acceptance_rate', 'driver_cancellation_rate',
    'booking_cancellation_rate', 'avg_fare', 'avg_distance',
    'driver_earnings', 'earnings_per_km', 'supply_gap', 'reliability_score'
]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, square=True,
    linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Matrix — Ward-Level Metrics', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 5.3 Outlier Detection (IQR Method)

In [ ]:
outlier_cols = [
    'driver_cancellation_rate', 'conversion_rate', 'earnings_per_km',
    'avg_fare', 'avg_distance', 'supply_gap', 'reliability_score'
]

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(22, 5))

for i, col in enumerate(outlier_cols):
    sns.boxplot(y=df[col], ax=axes[i], color=sns.color_palette('Set2')[i], width=0.4)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')

plt.suptitle('Outlier Detection — Box Plots for Key Metrics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify outliers per column using IQR method
outlier_report = []
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report.append({
        'Column': col,
        'Q1': round(Q1, 4), 'Q3': round(Q3, 4), 'IQR': round(IQR, 4),
        'Lower Fence': round(lower, 4), 'Upper Fence': round(upper, 4),
        'Outliers': n_outliers
    })

pd.DataFrame(outlier_report)

### 5.4 Top & Bottom Wards

In [ ]:
print('🏆 Top 10 Wards by Conversion Rate')
display(df.nlargest(10, 'conversion_rate')[['ward', 'conversion_rate', 'completed_trips', 'driver_earnings', 'reliability_score']])

print('\n⚠️ Bottom 10 Wards by Conversion Rate')
display(df.nsmallest(10, 'conversion_rate')[['ward', 'conversion_rate', 'completed_trips', 'driver_earnings', 'reliability_score']])

In [ ]:
print('🏆 Top 10 Wards by Reliability Score')
display(df.nlargest(10, 'reliability_score')[['ward', 'reliability_score', 'driver_cancellation_rate', 'driver_quote_acceptance_rate']])

print('\n⚠️ Bottom 10 Wards by Reliability Score')
display(df.nsmallest(10, 'reliability_score')[['ward', 'reliability_score', 'driver_cancellation_rate', 'driver_quote_acceptance_rate']])

### 5.5 Scatter — Driver Cancellation vs. Conversion Rate

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    df['driver_cancellation_rate'] * 100,
    df['conversion_rate'] * 100,
    c=df['earnings_per_km'],
    cmap='plasma', s=60, alpha=0.75, edgecolor='white', linewidth=0.5
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Earnings per km (₹)', fontsize=12)
ax.set_xlabel('Driver Cancellation Rate (%)', fontsize=13)
ax.set_ylabel('Conversion Rate (%)', fontsize=13)
ax.set_title('Driver Cancellation vs Conversion Rate (colored by ₹/km)', fontsize=15, fontweight='bold')

# Annotate extreme wards
for _, row in df.nlargest(3, 'driver_cancellation_rate').iterrows():
    ax.annotate(row['ward'], (row['driver_cancellation_rate']*100, row['conversion_rate']*100),
                fontsize=8, alpha=0.8)
for _, row in df.nlargest(3, 'conversion_rate').iterrows():
    ax.annotate(row['ward'], (row['driver_cancellation_rate']*100, row['conversion_rate']*100),
                fontsize=8, alpha=0.8)

plt.tight_layout()
plt.show()

### 5.6 Supply Gap vs Earnings

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    df['supply_gap'] / 1e5,
    df['driver_earnings'] / 1e7,
    c=df['driver_cancellation_rate'] * 100,
    cmap='YlOrRd', s=60, alpha=0.75, edgecolor='white', linewidth=0.5
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Driver Cancel Rate (%)', fontsize=12)
ax.set_xlabel('Supply Gap (in Lakhs)', fontsize=13)
ax.set_ylabel('Driver Earnings (in Crores ₹)', fontsize=13)
ax.set_title('Supply Gap vs Driver Earnings (colored by Cancel Rate)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save Cleaned Dataset

In [ ]:
df.to_csv('data/cleaned.csv', index=False)
print(f'✅ Saved cleaned data → data/cleaned.csv')
print(f'   Shape: {df.shape}')
print(f'   Columns: {list(df.columns)}')

In [ ]:
# Final verification
verify = pd.read_csv('data/cleaned.csv')
print(f'Reloaded shape: {verify.shape}')
verify.head()